## Notes de lecture

Ce notebook suit 4 étapes :

1. définir les points à visiter autour du dépôt de Rodez ;
2. appeler OSRM pour obtenir les matrices de distance et de durée ;
3. résoudre le problème de tournées avec OR-Tools ;
4. afficher les routes sur une carte Folium et exporter le résultat en HTML.


In [1]:
# Packages nécessaires au calcul d'itinéraires et à l'affichage de la carte
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import requests
import folium

# Liste des points à visiter.
# Le premier point correspond au dépôt, les autres sont les visites à répartir.
locations = [
    ("Depot", "Rodez", 2.5734, 44.3526),

    ("A", "Baraqueville", 2.4318, 44.2766),
    ("B", "Flavin", 2.6032, 44.2889),
    ("C", "Saint-Côme-d'Olt", 2.8140, 44.5150),
    ("D", "Estaing", 2.6710, 44.5540),
    ("E", "Conques", 2.3970, 44.5990),
    ("F", "Valady", 2.4270, 44.4550),
    ("G", "Nauviale", 2.4260, 44.5200),
    ("H", "Firmi", 2.3100, 44.5400),
    ("I", "Cransac", 2.2840, 44.5250),
    ("J", "Balsac", 2.4450, 44.4010),
    ("K", "Villefranche-de-Rouergue", 2.0370, 44.3510),
]

# OSRM attend des coordonnées sous la forme lon,lat séparées par des points-virgules.
coordinates = ";".join(
    f"{lon},{lat}"
    for _, _, lon, lat in locations
)

# Appel à l'API table d'OSRM pour récupérer les distances et les durées entre tous les points.
url = (
    "http://127.0.0.1:5001/table/v1/driving/"
    + coordinates
    + "?annotations=distance,duration"
)

print(url)


# Les réponses OSRM sont en mètres et en secondes, on arrondit pour simplifier le travail de l'optimiseur.
response = requests.get(url)
response.raise_for_status()

data = response.json()

distance_matrix = [
    [round(value) for value in row]
    for row in data["distances"]
]

duration_matrix = [
    [round(value) for value in row]
    for row in data["durations"]
]

# La matrice de distance sert de coût principal pour OR-Tools.
print(distance_matrix)
print(duration_matrix)


# Résolution du VRP avec 4 véhicules et un dépôt unique.
def solve_vrp():
    print("debut")
    
    vehicle_count = 4
    depot_index = 0

    manager = pywrapcp.RoutingIndexManager(
        len(distance_matrix),
        vehicle_count,
        depot_index,
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        return distance_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # DEBUT - si on enlève ce bloc, on se retrouve avec des véhocule qui font énormément de distance et d'autre presque aucune
    # en effet, sans ce bloc on minimize distance totale = distance véhicule 1 + distance véhicule 2 + ...
    # on évite avec ce bloc  qu’un véhicule ait une route beaucoup plus longue que les autres; 
    # OR-Tools essaie alors de minimiser le span, c’est-à-dire l’écart entre la route la plus courte et la plus longue, ou plus simplement le poids du véhicule qui fait le plus de distance.
    routing.AddDimension(
        transit_callback_index,
        0,
        200000,  # max distance per vehicle in meters
        True,
        "Distance",
    )
    
    distance_dimension = routing.GetDimensionOrDie("Distance")
    
    # Important: this makes OR-Tools balance routes
    distance_dimension.SetGlobalSpanCostCoefficient(100)
    # FIN
    

    for vehicle_id in range(vehicle_count):
        routing.solver().Add(
            routing.NextVar(routing.Start(vehicle_id)) != routing.End(vehicle_id)
        )
    
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    if solution is None:
        print("No solution found")
        return

    routes = []
    
    for vehicle_id in range(vehicle_count):
        index = routing.Start(vehicle_id)
    
        route = []
        route_distance = 0
    
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
    
            previous_index = index
            index = solution.Value(routing.NextVar(index))
    
            route_distance += routing.GetArcCostForVehicle(
                previous_index,
                index,
                vehicle_id,
            )
    
        route.append(manager.IndexToNode(index))
    
        print(f"Vehicle {vehicle_id}:")
        
        route_labels = [
            f"{locations[node][0]} ({locations[node][1]})"
            for node in route
        ]
        
        print(" -> ".join(route_labels))
        
        print(f"Distance: {route_distance / 1000:.2f} km")
        print()
        routes.append({
            "vehicle_id": vehicle_id,
            "nodes": route,
            "distance": route_distance,
        })
        
    return routes


# Récupère la géométrie exacte d'une tournée auprès d'OSRM pour l'afficher sur la carte.
def get_osrm_geometry(route_nodes):
    coordinates = ";".join(
        f"{locations[node][2]},{locations[node][3]}"
        for node in route_nodes
    )

    url = (
        "http://127.0.0.1:5001/route/v1/driving/"
        + coordinates
        + "?overview=full&geometries=geojson"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # OSRM returns [lon, lat], Folium needs [lat, lon]
    return [
        [lat, lon]
        for lon, lat in data["routes"][0]["geometry"]["coordinates"]
    ]


# Construit la carte Folium, place les points et trace chaque tournée avec une couleur dédiée.
def display_routes_on_map(routes):
    depot_lat = locations[0][3]
    depot_lon = locations[0][2]

    m = folium.Map(
        location=[depot_lat, depot_lon],
        zoom_start=10,
        tiles="CartoDB positron",
    )

    colors = ["blue", "red", "green", "purple", "orange", "darkred"]

    for name, city, lon, lat in locations:
        folium.Marker(
            location=[lat, lon],
            popup=f"{name} - {city}",
            tooltip=f"{name} ({city})",
            icon=folium.Icon(
                color="black" if name == "Depot" else "cadetblue",
                icon="home" if name == "Depot" else "info-sign",
            ),
        ).add_to(m)

    for route in routes:
        vehicle_id = route["vehicle_id"]
        route_nodes = route["nodes"]
        color = colors[vehicle_id % len(colors)]

        geometry = get_osrm_geometry(route_nodes)

        folium.PolyLine(
            geometry,
            color=color,
            weight=5,
            opacity=0.8,
            tooltip=f"Vehicle {vehicle_id} - {route['distance'] / 1000:.2f} km",
        ).add_to(m)

    return m


# Lance l'optimisation puis exporte la carte finale dans vrp_routes.html.
if __name__ == "__main__":
    routes = solve_vrp()
    print("display")
    m = display_routes_on_map(routes)
    m
    m.save("vrp_routes.html")

http://127.0.0.1:5001/table/v1/driving/2.5734,44.3526;2.4318,44.2766;2.6032,44.2889;2.814,44.515;2.671,44.554;2.397,44.599;2.427,44.455;2.426,44.52;2.31,44.54;2.284,44.525;2.445,44.401;2.037,44.351?annotations=distance,duration
[[0, 20886, 9764, 35168, 36883, 37750, 19874, 26568, 34386, 36275, 16866, 59999], [19294, 0, 14932, 52640, 54356, 51348, 27853, 38086, 42364, 38708, 21442, 41336], [9630, 15034, 0, 46346, 48062, 48928, 30661, 37746, 45173, 47062, 27653, 56306], [35596, 53280, 46900, 0, 13942, 54067, 45664, 42885, 56490, 58379, 49261, 92394], [36787, 54471, 48091, 13678, 0, 40392, 30477, 29210, 42815, 44704, 38061, 74689], [40644, 58329, 51948, 52453, 39043, 0, 24510, 14734, 23695, 25585, 35942, 68113], [19739, 27838, 30168, 45346, 30470, 21504, 0, 10323, 14918, 16808, 9450, 46002], [26337, 44021, 37641, 42531, 29120, 11721, 10202, 0, 14144, 16034, 19562, 54414], [33305, 41404, 43734, 55318, 41908, 19796, 14017, 13214, 0, 4618, 23016, 41807], [35948, 39129, 46378, 57962, 44551, 2